# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR2)

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields by their @id
print("Available record sets (@id and name):")
for rs in metadata.record_sets:
    print(f"  @id: {rs.id}\n    name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into DataFrames, using record set @id
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

example_record_set_id = record_sets_ids[0] if len(record_sets_ids) > 0 else None
if example_record_set_id and not dataframes[example_record_set_id].empty:
    print(f"Columns for record set {example_record_set_id}:\n", dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No dataframes loaded or the dataset is empty.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Let's pick the main record set (assume the first record set is the main table)

main_record_set_id = example_record_set_id
df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()

# Find a numeric field to analyze (heuristically: first field with int/float values)
numeric_field_id = None
if not df.empty:
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break

if numeric_field_id:
    print(f"Numeric field selected for analysis: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # Use mean as a demo threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where `{numeric_field_id}` > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    )
    print(f"\nNormalized `{numeric_field_id}` for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by the first categorical field
    group_field_id = None
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]) and c != numeric_field_id:
            group_field_id = c
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of `{numeric_field_id}` by `{group_field_id}`:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f'Distribution of `{numeric_field_id}`')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If a group field was found, plot boxplot
    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

This notebook demonstrated how to load and explore the FAIR^2 clinical dataset using the `mlcroissant` library. We reviewed the dataset's structure via its Croissant schema, inspected available record sets and fields by their `@id`, loaded the tabular data, applied basic EDA including filtering and normalization, and visualized key numeric field distributions. This approach can be extended to deeper statistical or modeling tasks using the rich metadata and data in Croissant-compliant datasets.